# 01 — SpaceX API : collecte des lancements Falcon 9

**Objectif :** reconstruire la collecte à partir de la [SpaceX REST API v4](https://github.com/r-spacex/SpaceX-API). Ce notebook interroge les données accessibles au moment de l’exécution : elles peuvent différer du jeu historique IBM (90 lancements). Les tableaux et chiffres affichés ne sont valides **qu’après exécution**.

In [1]:
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

pd.set_option("display.max_columns", 30)
OUT = Path("data") if Path("data").exists() else Path("../data")
OUT.mkdir(parents=True, exist_ok=True)
BASE = "https://api.spacexdata.com/v4"

def get_json(endpoint):
    response = requests.get(f"{BASE}/{endpoint}", timeout=40)
    response.raise_for_status()
    return response.json()

In [2]:
launches = get_json("launches/past")
rockets = get_json("rockets")
launchpads = get_json("launchpads")
landpads = get_json("landpads")
print("Date de collecte UTC :", datetime.now(timezone.utc).isoformat())
print("Lancements passés obtenus :", len(launches))
print("Premier lancement récupéré :", launches[0].get("date_utc") if launches else "aucun")

HTTPError: 522 Server Error: <none> for url: https://api.spacexdata.com/v4/launches/past

In [ ]:
rockets_by_id = {r["id"]: r.get("name") for r in rockets}
launchpads_by_id = {p["id"]: p.get("name") for p in launchpads}
landpads_by_id = {p["id"]: p.get("name") for p in landpads}

records = []
for launch in launches:
    rocket_name = rockets_by_id.get(launch.get("rocket"))
    if rocket_name != "Falcon 9":
        continue
    cores = launch.get("cores") or []
    first_core = cores[0] if cores else {}
    records.append({
        "FlightNumber": launch.get("flight_number"),
        "Date": launch.get("date_utc"),
        "Mission": launch.get("name"),
        "Rocket": rocket_name,
        "LaunchSite": launchpads_by_id.get(launch.get("launchpad")),
        "Success": launch.get("success"),  # succès de mission ≠ succès d’atterrissage
        "LandingSuccess": first_core.get("landing_success"),
        "LandingType": first_core.get("landing_type"),
        "LandingPad": landpads_by_id.get(first_core.get("landpad")),
        "Reused": first_core.get("reused"),
        "CoreFlight": first_core.get("flight"),
        "FlightID": launch.get("id"),
    })
df = pd.DataFrame(records).sort_values("Date").reset_index(drop=True)
print("Lancements Falcon 9 retenus :", len(df))
display(df.head(5))

In [ ]:
print("Dimensions :", df.shape)
display(df.isna().sum().rename("Missing values").to_frame())
print("Atterrissage indiqué :", df["LandingSuccess"].value_counts(dropna=False).to_dict())

In [ ]:
output_file = OUT / "spacex_falcon9_api_current.csv"
df.to_csv(output_file, index=False)
print("Fichier sauvegardé :", output_file.resolve())

## Interprétation à rédiger après exécution
- Combien de lancements Falcon 9 sont présents dans **cette extraction** ?
- Combien de valeurs manquent dans `LandingSuccess` et `LandingPad` ?
- **Attention :** `Success` indique la réussite du lancement/de la mission, et **non** celle de l’atterrissage du premier étage.
- Pour retrouver les valeurs exactes du quiz IBM, employer les **datasets historiques du cours**, plutôt que les données live.